## Extractor Configuration and Testing ##

This is the notebook for configuring and testing the Extractor. 

Please first consider (together with Gemini) which categories need to be captured so that your test orders are recorded accurately. 

Minimum requirements: It should be possible to process orders containing multiple pizzas. 

Please start by using Anthropic Haiku via the API; later, you can try to see if local models also produce good results. 


In [3]:
from dotenv import load_dotenv
import os
import getpass

load_dotenv()


if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Introduce tu  AWS Key: ")

In [4]:
from langchain.chat_models import init_chat_model

model = init_chat_model("laude-haiku-4-5-20251001", model_provider="anthropic", temperature=0.0)

### Here is the basic structure for the Extractor: ###

We need a **system prompt** that explains to the model what it needs to do.

The respective order is provided as **Human Massage**. 

In [5]:
#Example for a basic system prompt
from langchain_core.prompts import ChatPromptTemplate

model = init_chat_model("claude-haiku-4-5-20251001", model_provider="", temperature=0.0)

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert extraction algorithm. "
                "Only extract relevant information from the text. "
                "A customer order may contain one or multiple pizzas — extract each one separately. "
                "Parse the customer's pizza order into valid JSON with a top-level key 'order', "
                "which is a list of pizzas, each with the keys pizza_name, pizza_size, and pizza_ingredientes."
        ),
        #MessagesPlaceholder("examples"),
        
        ("human", "{order_text}"),
    ]
)

In [4]:
order = "I'll take two pizzas: a medium with pepperoni and mushrooms, and a small margherita, please."
prompt = prompt_template.invoke({"order_text": order})
res = model.invoke(prompt)
res.content


'```json\n{\n  "order": [\n    {\n      "pizza_name": "Pepperoni and Mushroom",\n      "pizza_size": "medium",\n      "pizza_ingredientes": ["pepperoni", "mushrooms"]\n    },\n    {\n      "pizza_name": "Margherita",\n      "pizza_size": "small",\n      "pizza_ingredientes": ["tomato", "mozzarella", "basil"]\n    }\n  ]\n}\n```'

Then we need to define the **extraction schema**. Try using **Pydantic** first, then try to use a **JSON Schema** with mode=json (preferably converted by Gemini).

Here is a simple schema to get started:

In [5]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

class Pizza(BaseModel):
    """Information about a the ordered pizza."""
    
    pizza_name: str = Field(
        ..., description="The name of the pizza if provided, else None."
    )
    pizza_size: Literal["small", "medium", "normal", "large"] = Field(
        ..., description="The size of the pizza"
    )
    pizza_ingredientes: Optional[str] = Field(
        default=None, description="The ingredients for the pizza. In case of a half-half pizza, the output should be Half: ingredients or name of the first half, Half: ingredients or name of the second half"
    )
    quantity: int = Field(
        default=1, description="How many of this exact pizza the customer ordered."
    )


class Pizza_order(BaseModel):
    """Extracted data about pizzas."""
    order: List[Pizza]


structured_llm = model.with_structured_output(schema=Pizza_order)



In [6]:
order = "I'd like three large pepperoni pizzas and one medium veggie."
prompt = prompt_template.invoke({"order_text": order})
res = structured_llm.invoke(prompt)
res.model_dump()

{'order': [{'pizza_name': 'Pepperoni',
   'pizza_size': 'large',
   'pizza_ingredientes': None,
   'quantity': 3},
  {'pizza_name': 'Veggie',
   'pizza_size': 'medium',
   'pizza_ingredientes': None,
   'quantity': 1}]}

If necesary, you can add more examples to the prompt template to improve the extraction accuracy.

In [7]:
#Creat a list of few-shot massege with the original examples form the Google Whitepaper

examples = [
    {"role": "user", "content": "I want a small pizza with cheese, tomato sauce, and pepperoni."},
    {"role": "assistant", "content": '{"size": "small","type": "normal", "ingredients": [["cheese", "tomato sauce", "peperoni"]]}'},
    {"role": "user", "content": "Can I get a large pizza with tomato sauce, basil and mozzarella?"},
    {"role": "assistant", "content": '{"size": "large","type": "normal","ingredients": [["tomato sauce", "bazel", "mozzarella"]]}'},
]    

In [8]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert extraction algorithm. "
            "Only extract relevant information from the text. "
            "Parse a customer's pizza order into valid JSON with the keys name, size, ingredients."
        ),
        #MessagesPlaceholder("examples"),
        
        ("human", "{order_text}"),
    ]
)

In [ ]:
text = "I want a small pizza with cheese, tomato sauce, and pepperoni. "
prompt = prompt_template_with_examples.invoke({"examples": examples, "order_text": text})

for message in prompt.messages:
    message.pretty_print()

In [ ]:
res = model.invoke(prompt)
res.pretty_print()

================================== Ai Message ==================================

```json
{
  "size": "small",
  "type": "normal",
  "ingredients": ["cheese", "tomato sauce", "pepperoni"]
}
```
